# O clima ajuda a prever a dengue?

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AknoLeonel/DataAn/blob/main/notebooks/estudo-dengue-clima.ipynb)

Estudo de caso do [DataAn](https://dataan.vercel.app/dados/estudo-dengue-clima/). Ele roda o mesmo
código que gera a página do site, com dados ao vivo do [InfoDengue](https://info.dengue.mat.br/)
(Fiocruz/FGV).

**Pergunta.** Temperatura e umidade das semanas anteriores melhoram a previsão de casos de dengue
das próximas 4 semanas, além do que a própria série de casos já informa?

**Por que importa.** O mosquito *Aedes aegypti* se reproduz mais rápido no calor úmido, com atraso de
semanas até virar caso notificado. Se o clima antecipa a curva, uma secretaria de saúde ganha tempo
para agir.

**Critério definido antes de ver os dados.** Um modelo só substitui o atual se reduzir o erro da
previsão ingênua (repetir a última semana) em pelo menos **15%** em pelo menos **4 das 6** capitais.

**Validação.** Origem móvel: em cada uma das últimas 52 semanas, cada modelo é ajustado só com o
passado e prevê de 1 a 4 semanas à frente. Nada do futuro vaza para o ajuste.

In [ ]:
# No Colab (ou num ambiente novo): instala o pacote do DataAn direto do GitHub.
%pip install -q "dataan @ git+https://github.com/AknoLeonel/DataAn.git#subdirectory=pipelines" matplotlib

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dataan.analysis.experiment import HOLT, MODELS, NAIVE, RIDGE, RIDGE_CLIMA, backtest
from dataan.datasets.saude import CAPITALS, CRITERIO_CAPITAIS, CRITERIO_GANHO
from dataan.sources import infodengue

pd.set_option("display.precision", 1)
ANO = date.today().year

## 1. Os dados

Uma linha por semana epidemiológica e capital: casos estimados (`casos_est`, corrigidos pelo atraso
de notificação) e o clima semanal das estações usadas pelo InfoDengue (`tempmed`, `umidmed`).

In [ ]:
dados = {
    cidade: infodengue.alertcity(geocodigo, "dengue", ANO - 4, ANO)
    for cidade, _uf, geocodigo in CAPITALS
}
resumo = pd.DataFrame(
    {
        cidade: {
            "semanas": len(df),
            "de": df["date"].min().date(),
            "até": df["date"].max().date(),
            "clima preenchido (%)": 100 * df[["tempmed", "umidmed"]].notna().all(axis=1).mean(),
            "casos/semana (mediana)": df["casos_est"].median(),
        }
        for cidade, df in dados.items()
    }
).T
resumo

In [ ]:
fig, axes = plt.subplots(len(dados), 1, figsize=(10, 2.2 * len(dados)), sharex=True)
for ax, (cidade, df) in zip(axes, dados.items()):
    ax.plot(df["date"], df["casos_est"], color="#d1495b", lw=1.4)
    ax.set_ylabel("casos", color="#d1495b")
    ax.set_title(cidade, loc="left", fontsize=10)
    ax2 = ax.twinx()
    ax2.plot(df["date"], df["tempmed"], color="#2e86ab", lw=0.9, alpha=0.7)
    ax2.set_ylabel("°C", color="#2e86ab")
fig.suptitle("Casos estimados de dengue (vermelho) e temperatura média semanal (azul)")
fig.tight_layout()

## 2. Olhar antes de modelar: correlação com defasagem

Correlação entre a temperatura de *k* semanas atrás e o log dos casos de hoje. Atenção: as duas séries
têm sazonalidade anual, então parte da correlação é só "verão com verão". É por isso que a pergunta
de verdade é respondida com modelos que **já** conhecem a sazonalidade (seção 3).

In [ ]:
defasagens = range(0, 13)
corr = pd.DataFrame(
    {
        cidade: [np.log1p(df["casos_est"]).corr(df["tempmed"].shift(k)) for k in defasagens]
        for cidade, df in dados.items()
    },
    index=pd.Index(defasagens, name="defasagem (semanas)"),
)
ax = corr.plot(figsize=(9, 4), marker="o", ms=3)
ax.axhline(0, color="gray", lw=0.8)
ax.set_ylabel("correlação")
ax.set_title("Temperatura de k semanas atrás vs. log dos casos")
corr.idxmax().rename("defasagem de maior correlação")

## 3. O experimento

Cinco modelos, mesma validação (ingênuo, sazonal ingênuo, Holt amortecido, regressão com defasagens, regressão com defasagens + clima).
As duas regressões são idênticas, exceto pelo clima: a diferença entre elas isola o efeito do clima.

`ganho = 1 − erro do modelo / erro do ingênuo`. Positivo é melhor que o ingênuo.

In [ ]:
resultados = {cidade: backtest(df, horizon=4, origins=52) for cidade, df in dados.items()}

ganho = pd.DataFrame({cidade: bt.skill for cidade, bt in resultados.items()}).T[MODELS]
(ganho.drop(columns=NAIVE) * 100).round(1)

In [ ]:
capitais_no_criterio = (ganho >= CRITERIO_GANHO).sum().drop(NAIVE)
vencedores = capitais_no_criterio[capitais_no_criterio >= CRITERIO_CAPITAIS]
print(f"Capitais com ganho ≥ {CRITERIO_GANHO:.0%}, por modelo:")
print(capitais_no_criterio.to_string())
print()
print("Atingem o critério:", ", ".join(vencedores.index) or "nenhum modelo")

In [ ]:
efeito = ((ganho[RIDGE_CLIMA] - ganho[RIDGE]) * 100).sort_values()
ax = efeito.plot.barh(color=np.where(efeito > 0, "#2a9d8f", "#c0c0c0"), figsize=(7, 3.5))
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("p.p. de ganho (regressão com clima − sem clima)")
ax.set_title("Quanto o clima mudou a qualidade da previsão")
print(f"Clima ajudou em {(efeito > 0).sum()} de {len(efeito)} capitais; efeito médio {efeito.mean():+.1f} p.p.")

## 4. Como as previsões se parecem

Na capital com mais casos: cada ponto das linhas tracejadas foi previsto **4 semanas antes**, só com
o que se sabia naquela data.

In [ ]:
cidade = max(resultados, key=lambda c: resultados[c].actual.iloc[-60:].sum())
bt = resultados[cidade]
janela = bt.preds[NAIVE].index
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(janela, bt.actual.reindex(janela), color="black", lw=2, label="real")
for modelo, cor in ((NAIVE, "#999999"), (HOLT, "#e9c46a"), (RIDGE, "#2e86ab"), (RIDGE_CLIMA, "#2a9d8f")):
    if modelo in bt.preds:
        ax.plot(bt.preds[modelo].index, bt.preds[modelo], "--", color=cor, label=modelo)
ax.set_title(f"{cidade}: previsão de 4 semanas vs. o que aconteceu")
ax.set_ylabel("casos estimados por semana")
ax.legend(fontsize=8)

## 5. Robustez: o resultado depende da janela?

Repetimos a comparação das duas regressões com 26, 52 e 78 semanas de avaliação. Uma conclusão que
muda de sinal com a janela não é uma conclusão.

In [ ]:
linhas = []
for origens in (26, 52, 78):
    for cidade, df in dados.items():
        b = backtest(df, horizon=4, origins=origens)
        if RIDGE_CLIMA in b.skill:
            linhas.append({"janela": origens, "cidade": cidade, "efeito_pp": 100 * (b.skill[RIDGE_CLIMA] - b.skill[RIDGE])})
robustez = pd.DataFrame(linhas).pivot(index="cidade", columns="janela", values="efeito_pp")
robustez.loc["média"] = robustez.mean()
robustez.round(1)

## 6. Conclusão

A resposta é recalculada toda semana com dados novos, por isso o texto abaixo é gerado a partir dos
números acima, e não escrito à mão. A mesma conclusão aparece na
[página do estudo](https://dataan.vercel.app/dados/estudo-dengue-clima/).

**Limitações.** O clima vem das estações usadas pelo InfoDengue (falhas são preenchidas com o valor
anterior); os casos das últimas semanas são estimativas que ainda mudam; 52 semanas cobrem uma única
temporada por capital. Um próximo passo natural é usar chuva acumulada e dados de reanálise
(ERA5) em vez das estações.

In [ ]:
ajudou = int((efeito > 0).sum())
if ajudou > len(efeito) / 2 and robustez.loc["média"].gt(0).all():
    print(f"Sim: o clima melhorou a previsão em {ajudou} de {len(efeito)} capitais, em todas as janelas testadas.")
elif ajudou > len(efeito) / 2:
    print(f"Em parte: o clima ajudou em {ajudou} de {len(efeito)} capitais, mas o efeito muda com a janela.")
else:
    print(f"Não de forma consistente: o clima ajudou em só {ajudou} de {len(efeito)} capitais "
          f"(efeito médio {efeito.mean():+.1f} p.p.). A série de casos já carrega quase toda a informação.")
print("Modelos que atingem o critério:", ", ".join(vencedores.index) or "nenhum")